# 06 — Data-quality gate (warn only)

Reads the Gold data-quality tables at the end of the daily build and writes a
small audit log. **It never fails the pipeline.** Warnings are printed and
persisted to `gold_pipeline_dq_log` so the operator can see them in the
Monitoring hub and the lakehouse.

**Checks**

| Check | Expected |
|---|---|
| Feature grid rows | 450 (5 stations × 90 dates) |
| Grid dates / stations | 90 / 5 |
| Observed station-days | 399 |
| Scored rows | at least 200 |
| Deviation rows | greater than 0 |
| Risk rows | 450 |
| Alerts reconcile | fired = Red + Amber |
| Latest risk date | equals the feature max date |
| Max absolute z | at most 25 (the cap) |
| Grid completeness | at least 80% |

A WARN means “look at this”, not “stop the build”.


In [ ]:
from datetime import datetime, timezone
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructField, StructType, StringType, TimestampType,
)

WORKSPACE_ID = "355cd573-8b17-430a-b606-c6b9424d36a1"
GOLD_ID = "df9a08e5-b370-4c2a-b608-6d0f326003ef"
GOLD = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{GOLD_ID}"


def read_gold(name: str):
    return spark.read.format("delta").load(f"{GOLD}/Tables/{name}")


def append_gold(name: str, sdf) -> None:
    path = f"{GOLD}/Tables/{name}"
    (sdf.write.format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .save(path))
    spark.sql(f"CREATE TABLE IF NOT EXISTS {name} USING DELTA LOCATION '{path}'")
    print(f"{name}: appended {sdf.count():,} rows")


feat = read_gold("gold_data_quality").select("metric", "value").toPandas()
risk = read_gold("gold_risk_data_quality").select("metric", "value").toPandas()
feat_map = dict(zip(feat["metric"], feat["value"]))
risk_map = dict(zip(risk["metric"], risk["value"]))

print(f"feature DQ metrics: {len(feat_map)}")
print(f"risk DQ metrics   : {len(risk_map)}")


In [ ]:
def fnum(mapping, key):
    try:
        return float(mapping.get(key))
    except (TypeError, ValueError):
        return None


def sval(mapping, key):
    v = mapping.get(key)
    return None if v is None else str(v)


checks = []


def add(name, actual, expected, ok, message):
    checks.append({
        "check_name": name,
        "actual": "" if actual is None else str(actual),
        "expected": str(expected),
        "status": "OK" if ok else "WARN",
        "message": message,
    })


feature_rows = fnum(feat_map, "feature_rows")
grid_dates = fnum(feat_map, "grid_dates")
grid_stations = fnum(feat_map, "grid_stations")
observed_rows = fnum(feat_map, "observed_rows")
scored_rows = fnum(feat_map, "scored_rows")
deviation_rows = fnum(feat_map, "deviation_rows")
risk_rows = fnum(risk_map, "risk_rows")
alerts_fired = fnum(risk_map, "alerts_fired")
alerts_red = fnum(risk_map, "alerts_red")
alerts_amber = fnum(risk_map, "alerts_amber")
latest_date = sval(risk_map, "latest_date")
max_date = sval(feat_map, "max_date")
min_date = sval(feat_map, "min_date")
max_abs_z = fnum(feat_map, "max_abs_z")
completeness = fnum(feat_map, "grid_completeness_pct")

add("feature_rows", feature_rows, 450, feature_rows == 450,
    "gold_entity_day_features must be 5 stations x 90 dates")
add("grid_dates", grid_dates, 90, grid_dates == 90,
    "the date grid must cover 90 days")
add("grid_stations", grid_stations, 5, grid_stations == 5,
    "the station grid must cover 5 stations")
add("observed_rows", observed_rows, 399, observed_rows == 399,
    "observed station-days in the seed")
add("scored_rows", scored_rows, ">= 200",
    scored_rows is not None and scored_rows >= 200,
    "enough scored days for the rules to be meaningful")
add("deviation_rows", deviation_rows, "> 0",
    deviation_rows is not None and deviation_rows > 0,
    "the deviation mart must not be empty")
add("risk_rows", risk_rows, 450, risk_rows == 450,
    "gold_risk_daily must be the full station-date grid")
add("alerts_reconcile", alerts_fired,
    None if alerts_red is None or alerts_amber is None else alerts_red + alerts_amber,
    alerts_fired is not None and alerts_red is not None
    and alerts_amber is not None
    and alerts_fired == alerts_red + alerts_amber,
    "alerts_fired must equal Red + Amber")
add("latest_date", latest_date, max_date, latest_date == max_date,
    "latest risk date should equal the feature max date")
add("min_date", min_date, "not null", bool(min_date),
    "the feature window must have a start date")
add("max_abs_z", max_abs_z, "<= 25",
    max_abs_z is not None and max_abs_z <= 25,
    "robust z must respect the cap")
add("grid_completeness_pct", completeness, ">= 0.8",
    completeness is not None and completeness >= 0.8,
    "at least 80% of the grid is observed")

checked_at = datetime.now(timezone.utc).replace(tzinfo=None)
batch_id = (risk_map.get("batch_id") or feat_map.get("batch_id") or "unknown")

log_rows = [
    (checked_at, str(batch_id), c["check_name"], c["actual"],
     c["expected"], c["status"], c["message"])
    for c in checks
]
schema = StructType([
    StructField("checked_at_utc", TimestampType(), False),
    StructField("batch_id", StringType(), False),
    StructField("check_name", StringType(), False),
    StructField("actual", StringType(), True),
    StructField("expected", StringType(), True),
    StructField("status", StringType(), False),
    StructField("message", StringType(), True),
])
log = spark.createDataFrame(log_rows, schema)
append_gold("gold_pipeline_dq_log", log)
display(log.toPandas())


In [ ]:
warnings = [c for c in checks if c["status"] == "WARN"]
print(f"DQ gate: {len(checks) - len(warnings)} OK, {len(warnings)} WARN")
for c in warnings:
    print(f"WARN {c['check_name']}: actual={c['actual']} expected={c['expected']} - {c['message']}")
print("Warn-only: the pipeline is not blocked.")
